# 03 · Question 3 — When should the signal be switched off?

A rule-based kill switch (`sv/validation/gate.py`, thresholds in `config.GATE_RULES`) turns a model
off for the coming week when any monitoring metric breaches its limit. The gate sees only information
available on the decision date.

**Champion** = the model run without a gate. **Challenger** = the same model, held in cash while the
gate is off, paying the full round-trip cost on every flip. The comparison is out-of-time from
`config.OOT_START` — the thresholds were fixed before looking at this window.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from sv import db

con = db.connect(read_only=True)
q   = lambda name, **p: db.run_sql_file(con, name, p or None)   # run sql/queries/<name>.sql
sql = lambda text, **p: db.read(con, text, p or None)                # run an inline query
pd.set_option("display.width", 140); plt.rcParams["figure.figsize"] = (10, 4)

In [ ]:
pd.DataFrame(config.GATE_RULES, index=["op", "threshold"]).T

## How often, and why, the gate is off

In [ ]:
sql("""SELECT model, COUNT(*) AS weeks, SUM((NOT gate_on)::INT) AS weeks_off,
              ROUND(AVG((NOT gate_on)::INT), 3) AS share_off FROM gate_decisions GROUP BY model ORDER BY model""")

In [ ]:
sql("""SELECT model, reason, COUNT(*) AS weeks FROM gate_decisions
       WHERE NOT gate_on GROUP BY model, reason ORDER BY model, weeks DESC""")

In [ ]:
g = sql("SELECT date, model, gate_on FROM gate_decisions").pivot(index="date", columns="model", values="gate_on")
ax = (~g).astype(int).plot(subplots=True, figsize=(11, 5), legend=True, yticks=[0, 1], title="1 = gate OFF")
plt.tight_layout(); plt.show()

## Champion vs challenger, out-of-time

In [ ]:
cc = q("champion_challenger", start=config.OOT_START).set_index("strategy").round(3); cc

In [ ]:
rows = []
for m in ["momentum", "gbm_expected", "gbm"]:
    a, b = cc.loc[m], cc.loc[f"{m}_gated"]
    rows.append({"model": m, "cagr_champion": a.cagr, "cagr_challenger": b.cagr, "return_cost": b.cagr - a.cagr,
                 "maxdd_champion": a.max_drawdown, "maxdd_challenger": b.max_drawdown, "dd_improvement": b.max_drawdown - a.max_drawdown,
                 "sharpe_champion": a.sharpe, "sharpe_challenger": b.sharpe})
pd.DataFrame(rows).set_index("model").round(3)

In [ ]:
pr = sql("SELECT date, strategy, ret_net FROM portfolio_returns WHERE date >= $s", s=config.OOT_START)
eq = (1 + pr.pivot(index="date", columns="strategy", values="ret_net").fillna(0)).cumprod()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, m in zip(axes, ["momentum", "gbm_expected", "gbm"]):
    eq[[m, f"{m}_gated", "universe_ew"]].plot(ax=ax, title=f"{m}: champion vs challenger (OOT)"); ax.set_ylabel("growth of 1")
plt.tight_layout(); plt.show()